![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 18 -- Lab 1: Arabic Text Search with AraBERT (Mini-RAG)

**Scenario:** A Saudi news agency receives hundreds of Arabic articles daily. Journalists need a tool to type a query in Arabic and instantly find the most relevant articles -- even if the exact words don't match. Your job is to build a **semantic search engine** using AraBERT embeddings and cosine similarity.

This is the **retrieval** half of RAG (Retrieval-Augmented Generation) -- the technique behind modern AI assistants that look up information before answering.

| Part | Goal |
|---|---|
| Part 1 | Setup: install libraries and load AraBERT |
| Part 2 | Explore the Arabic document corpus |
| Part 3 | Convert text to embeddings with BERT |
| Part 4 | Embed all documents into a matrix |
| Part 5 | Build the search function with cosine similarity |
| Part 6 | Test the search engine with Arabic queries |
| Part 7 | Compare AraBERT vs multilingual BERT |
| Part 8 | (Bonus) Interactive search loop |

---

## Setup

In [ ]:
!pip install transformers torch -q

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---

## Part 1: Load AraBERT

AraBERT is a BERT model pre-trained specifically on Arabic text (Wikipedia, news, books). It understands Arabic much better than a generic multilingual model because it was trained exclusively on Arabic.

We use `aubmindlab/bert-base-arabertv02` -- the most widely used version. No special preprocessing needed (unlike older AraBERT versions).

In [ ]:
# --- GIVEN ---
MODEL_NAME = "aubmindlab/bert-base-arabertv02"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f"Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Max sequence length: {tokenizer.model_max_length}")

Let's see how AraBERT tokenizes Arabic text:

In [ ]:
# --- GIVEN ---
sample = "القطة جلست على السجادة"
tokens = tokenizer.tokenize(sample)
print(f"Text:   {sample}")
print(f"Tokens: {tokens}")
print(f"IDs:    {tokenizer.encode(sample)}")

---

## Part 2: The Arabic Document Corpus

Our news agency has a collection of short articles. Each covers a different topic: science, sports, technology, culture, health, and more.

In [ ]:
# --- GIVEN ---
documents = [
    "اكتشف العلماء كوكبا جديدا خارج المجموعة الشمسية يحتوي على ماء سائل على سطحه مما يزيد احتمال وجود حياة",
    "فاز المنتخب السعودي لكرة القدم بكأس آسيا بعد مباراة مثيرة انتهت بركلات الترجيح",
    "أطلقت شركة سعودية ناشئة تطبيقا جديدا للذكاء الاصطناعي يساعد المزارعين على مراقبة محاصيلهم باستخدام صور الأقمار الصناعية",
    "افتتح متحف جديد في الرياض يعرض قطعا أثرية نادرة من حضارة دلمون التي ازدهرت قبل آلاف السنين",
    "أظهرت دراسة طبية حديثة أن المشي ثلاثين دقيقة يوميا يقلل خطر الإصابة بأمراض القلب بنسبة أربعين بالمئة",
    "نثجحت تجربة إطلاق صاروخ فضائي سعودي من قاعدة الإطلاق الجديدة في منطقة تبوك",
    "حقق فريق الهلال السعودي لقب دوري أبطال آسيا للمرة الثالثة في تاريخه",
    "طور باحثون في جامعة الملك عبدالله للعلوم والتقنية روبوتا يمكنه السباحة تحت الماء لدراسة الشعاب المرجانية في البحر الأحمر",
    "أعلنت وزارة الثقافة عن مهرجان سينمائي دولي يقام في مدينة جدة ويستضيف مخرجين من عشرين دولة",
    "اكتشف فريق بحثي سعودي طريقة جديدة لتحلية مياه البحر باستخدام الطاقة الشمسية بتكلفة منخفضة",
    "سجل لاعب كرة السلة السعودي رقما قياسيا جديدا في عدد النقاط المسجلة في مباراة واحدة",
    "بدأت المملكة العربية السعودية في بناء أكبر مشروع للطاقة الشمسية في العالم ضمن رؤية ألفين وثلاثين",
    "نظمت جامعة الملك سعود مؤتمرا دوليا حول استخدام الذكاء الاصطناعي في التعليم الجامعي",
    "أثبتت دراسة علمية أن النوم الكافي يحسن الذاكرة ويساعد الطلاب على التحصيل الدراسي بشكل أفضل",
    "افتتحت مكتبة الملك فهد الوطنية قسما جديدا للمخطوطات العربية القديمة والنادرة",
    "تمكن علماء الفلك من رصد ثقب أسود هائل في مركز مجرة قريبة من درب التبانة",
    "أعلنت الهيئة العامة للرياضة عن خطة لبناء عشرة ملاعب رياضية جديدة في مختلف مناطق المملكة",
    "طور مهندسون سعوديون سيارة كهربائية تعمل بالطاقة الشمسية وتقطع مسافة خمسمئة كيلومتر بشحنة واحدة",
    "استضافت المملكة العربية السعودية بطولة العالم للشطرنج التي شارك فيها أبطال من خمسين دولة",
    "أطلقت هيئة الفضاء السعودية قمرا صناعيا جديدا لمراقبة التغيرات المناخية في منطقة الخليج العربي"
]

print(f"Corpus size: {len(documents)} documents\n")
for i, doc in enumerate(documents):
    print(f"[{i:2d}] {doc[:80]}...")

---

## Part 3: Convert Text to Embeddings

To search semantically, we need to convert each document into a **vector** (a list of numbers) that captures its meaning. BERT gives us a vector for each token. To get a single vector for the whole text, we **average** all token vectors (mean pooling).

### Task 1: Implement `get_embedding`

**TODO:**
1. Tokenize the text using `tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True)`
2. Move inputs to the correct device
3. Pass through the model inside `torch.no_grad()`
4. Get `outputs.last_hidden_state` (shape: batch, seq_len, hidden_dim)
5. Use the `attention_mask` to compute a **mean** over only the real tokens (not padding)
6. Return the embedding as a 1D tensor

**Hint for mean pooling:** Expand the attention mask to match the hidden state shape, multiply element-wise, sum along the sequence dimension, and divide by the mask sum.

In [ ]:
def get_embedding(text, tokenizer, model):
    """Convert a text string into a single embedding vector using mean pooling."""
    # Your code here
    pass

In [ ]:
# Test your function
test_emb = get_embedding("القطة جلست على السجادة", tokenizer, model)
print(f"Embedding shape: {test_emb.shape}")    # Should be (768,)
print(f"First 5 values: {test_emb[:5]}")
print(f"Norm: {torch.norm(test_emb):.4f}")

---

## Part 4: Embed All Documents

### Task 2: Build the Document Embedding Matrix

**TODO:**
- Loop over all documents and compute their embeddings using `get_embedding`
- Stack them into a single matrix of shape `(num_docs, 768)`
- Normalize each embedding to unit length (divide by its L2 norm) so cosine similarity becomes a simple dot product

In [ ]:
# Your code here


---

## Part 5: Build the Search Function

### Task 3: Implement `search`

**Cosine similarity** measures how similar two vectors are:
- 1.0 = identical direction (same meaning)
- 0.0 = unrelated
- -1.0 = opposite meaning

Since we normalized our embeddings, cosine similarity = dot product.

**TODO:**
1. Compute the query embedding using `get_embedding`
2. Normalize it to unit length
3. Compute cosine similarity between the query and all documents (matrix multiplication)
4. Sort by similarity (highest first)
5. Return the top `k` results as a list of `(doc_index, similarity_score)` tuples

In [ ]:
def search(query, doc_embeddings, documents, tokenizer, model, top_k=5):
    """Search for the most relevant documents given an Arabic query."""
    # Your code here
    pass

---

## Part 6: Test the Search Engine

### Task 4: Search with Arabic Queries

**TODO:**
- Try each of the queries below
- For each result, print the rank, similarity score, and document text
- Check: do the top results make sense?

In [ ]:
queries = [
    "اكتشافات فضائية جديدة",          # space discoveries
    "رياضة كرة القدم في السعودية",     # football in Saudi
    "تقنية الذكاء الاصطناعي",          # AI technology
    "صحة الإنسان والطب",              # human health and medicine
    "الثقافة والفنون",                 # culture and arts
    "الطاقة المتجددة",                 # renewable energy
]

# Your code here


### Task 5: Try Your Own Queries

**TODO:** Write 3 Arabic queries of your own and test them. Try queries that:
- Use different words than the documents (does semantic search still work?)
- Are short (1-2 words) vs long (a full question)
- Are on a topic not well covered by the corpus

In [ ]:
# Your code here


---

## Part 7: AraBERT vs Multilingual BERT

Does a model trained specifically on Arabic really do better than Google's multilingual BERT (trained on 104 languages)?

### Task 6: Load Multilingual BERT and Compare

**TODO:**
1. Load `bert-base-multilingual-cased` (tokenizer + model)
2. Embed the same documents using multilingual BERT
3. Run the same queries on both models
4. Compare the top-3 results side by side -- which model returns more relevant documents?

In [ ]:
# Your code here


---

## Part 8: (Bonus) Interactive Search

### Task 7: Build an Interactive Search Loop

**TODO:** Create a loop that:
1. Asks the user to type an Arabic query using `input()`
2. Runs the search and displays top-3 results
3. Repeats until the user types "exit"

In [ ]:
# Your code here


---

## Discussion

1. Did the search engine find relevant documents even when the query used **different words** than the document? Why can it do this?
2. How did AraBERT compare to multilingual BERT? Which returned more relevant results for Arabic queries?
3. What happens when you search for a topic not covered by any document? What similarity scores do you get?
4. In a real RAG system, what would happen **after** retrieval? (Hint: the retrieved documents would be passed to a generative model like GPT.)
5. What are the limitations of this approach? (Think about corpus size, embedding quality, speed.)

---

## Wrap-Up

**What you learned:**
- How to use a pre-trained BERT model (AraBERT) to convert Arabic text into embedding vectors
- Mean pooling: averaging token embeddings to get a single sentence embedding
- Cosine similarity: measuring how close two embeddings are in meaning
- Building a semantic search engine -- the "R" in RAG
- Why language-specific models (AraBERT) outperform multilingual models on their target language

**Next:** Lab 2 -- Build a chatbot with Gemma-4!